# Flickr8k 数据集预处理

本笔记本将引导你完成 Flickr8k 图像描述（image captioning）任务的数据集预处理步骤。我们将按照以下步骤进行：

1. 了解数据集结构
2. 安装必要的依赖
3. 准备数据集
4. 运行预处理脚本
5. 验证预处理结果

## 1. 了解数据集结构

Flickr8k 数据集包含：
- 8,000 张图像
- 每张图像有 5 个描述
- 数据被分为训练集（6,000 张）、验证集（1,000 张）和测试集（1,000 张）

数据集的标准分割由 Karpathy 提供，我们将使用他的 JSON 文件来获取图像路径和描述。

## 2. 安装必要的依赖

在你的环境中安装必要的依赖。你可以使用以下命令安装：
```bash
pip install -r requirements.txt
```

## 3. 准备数据集

首先，你需要下载 [Flickr8k 数据集](https://www.kaggle.com/datasets/adityajn105/flickr8k) 和 [Karpathy 分割](http://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip)。

下载完成后，按照以下结构组织文件：
```
flickr8k/
├── Images/          # 存放所有图像
├── captions.txt    # 原始标注文件
└── dataset_flickr8k.json  # Karpathy 分割文件
```

运行以下脚本查看数据集内容：

In [ ]:
import json
from pathlib import Path
from PIL import Image

DATASET_ROOT = Path("data/flickr8k/")

with open(DATASET_ROOT / "dataset_flickr8k.json", "r") as f:
    data = json.load(f)

print("First 2 images:")
for i in range(2):
    image_data = data["images"][i]
    image = Image.open(DATASET_ROOT / "Images" / image_data["filename"])
    print(f"Image {i} ({image_data['filename']}, {image.size[0]}px x {image.size[1]}px):")
    image.show()
    print("Captions:")
    for caption in image_data["sentences"]:
        print(caption["raw"])
    print()

## 4. 数据预处理

Karpathy JSON 文件中包含了图像路径和描述信息，其中每个图像有5条描述。数据集已分为训练集、验证集和测试集。图像描述需将句子中词元 (token) 映射到词表 (wordmap / vocabulary) 中的索引。

因此，我们需要进行以下预处理步骤：

1. **创建词表**：统计词频，过滤低频词，添加特殊标记（\<start\>, \<end\>, \<unk\>, \<pad\>）。
2. **编码描述**：将描述转换为词表中的索引，添加开始和结束标记，填充到最大长度（数据加载时进行）。这些编码后的描述将保存为新的 JSON 文件。

In [ ]:
from collections import Counter
from tqdm import tqdm

min_word_freq = 5  # 最小词频，用于过滤掉不常用的词，减小词表大小


def build_vocab(karpathy_data: dict) -> dict[str, int]:
    """从Karpathy数据集构建词表

    Args:
        karpathy_data (dict): 包含图像标注数据的字典

    Returns:
        dict[str, int]: 词表，键为词，值为词在表中的索引
    """
    # 统计每个词的出现次数
    counter = Counter[str]()
    for image_data in tqdm(karpathy_data["images"], desc="Building vocab"):
        for caption in image_data["sentences"]:
            counter.update(caption["tokens"])

    words = [w for w in counter if counter[w] >= min_word_freq]
    words.sort(key=lambda x: counter[x], reverse=True)

    print(
        f"Vocab: total {len(counter)} words, clamped to {len(words)} words with "
        f"min_freq={min_word_freq}"
    )

    # 创建词表
    word_map = {}

    word_map["<pad>"] = 0
    word_map["<start>"] = 1
    word_map["<end>"] = 2
    word_map["<unk>"] = 3

    for w in words:
        word_map[w] = len(word_map)

    return word_map


def encode_captions(karpathy_data: dict, word_map: dict[str, int]):
    """Encode captions in Karpathy dataset to indices in word_map.

    Args:
        karpathy_data (dict): 包含图像标注数据的字典
        word_map (dict[str, int]): 词表，键为词，值为词在表中的索引

    Returns:
        list[dict]: 编码后的数据，每个元素是一个字典，包含图像文件名和编码后的句子列表。
    """
    # if max_len < 0:
    #     for image_data in karpathy_data["images"]:
    #         for caption in image_data["sentences"]:
    #             max_len = max(max_len, len(caption["tokens"]))
    #     max_len += 2  # Include <start> and <end>

    token_start = word_map["<start>"]
    token_end = word_map["<end>"]
    token_unk = word_map["<unk>"]  # Unknown
    encoded_data = []
    for image_data in karpathy_data["images"]:
        obj = {
            "filename": image_data["filename"],
            "split": image_data["split"],
            "sentences": [],
        }
        for caption in image_data["sentences"]:
            sentence = [token_start]
            for w in caption["tokens"]:
                sentence.append(word_map.get(w, token_unk))
            sentence.append(token_end)
            obj["sentences"].append(sentence)
        encoded_data.append(obj)
    return encoded_data

现在，让我们运行预处理脚本：

In [ ]:
word_map = build_vocab(data)
print(f"Total words in vocab (including special tokens): {len(word_map)}")
print(f"First 10 words: {', '.join(list(word_map.keys())[4:14])}")

# Save vocab to file
path = DATASET_ROOT / "vocab.json"
with open(path, "w") as f:
    json.dump(word_map, f)
print(f"✅ Vocab saved to {path}")

# Map caption tokens to indices
encoded_data = encode_captions(data, word_map)
path = DATASET_ROOT / "captions_encoded.json"
with open(path, "w") as f:
    json.dump(encoded_data, f)
print(f"✅ Encoded captions saved to {path}")

通过本笔记本，你已经完成了 Flickr8k 数据集的预处理步骤。在后续的作业中，你将使用这些预处理后的数据来训练和评估图像描述模型。